# ViT 추론 데모 & 어텐션 시각화
## 01_infer_vit.ipynb

**목표**: timm ViT-B/16 모델을 사용한 추론 데모와 어텐션 메커니즘 시각화

**산출물**:
- `preds_topk.csv` - Top-K 예측 결과
- `attention_overlay.png` - 어텐션 집중 영역 오버레이
- `patch_grid.png` - 패치 분할 시각화

**소요시간**: ~5분


In [ ]:
# 필수 라이브러리 임포트 및 설정
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
sys.path.append('..')
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../reports/tables', exist_ok=True)

# 핵심 라이브러리
import torch
import torch.nn.functional as F
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from typing import List, Tuple, Dict, Optional
import json
from tqdm import tqdm
from pathlib import Path

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

# 랜덤 시드 고정 (재현성)
torch.manual_seed(42)
np.random.seed(42)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 디바이스: {device}")
print(f"📦 PyTorch 버전: {torch.__version__}")
print(f"🔧 timm 버전: {timm.__version__}")

# GPU 메모리 정보 (CUDA 사용 시)
if device.type == 'cuda':
    print(f"🎮 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


ModuleNotFoundError: No module named 'timm'

In [ ]:
# ViT 모델 로드 및 설정
print("📥 ViT-Base/16 모델 로드 중...")

# timm에서 사전훈련된 ViT-B/16 모델 로드
model_name = 'vit_base_patch16_224.augreg_in21k_ft_in1k'
model = timm.create_model(model_name, pretrained=True)
model = model.to(device)
model.eval()

# 모델 정보 출력
print(f"✅ 모델 로드 완료: {model_name}")
print(f"📊 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"🔧 임베딩 차원: {model.embed_dim}")
print(f"🎯 패치 크기: {model.patch_embed.patch_size}")
print(f"📐 입력 크기: {model.patch_embed.img_size}")

# 데이터 전처리 파이프라인 설정
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

print(f"🔄 전처리 설정: {data_config}")

# ImageNet 클래스 레이블 로드
with open('https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt') as f:
    imagenet_classes = [line.strip() for line in f.readlines()]
    
print(f"📚 ImageNet 클래스 수: {len(imagenet_classes)}")


In [ ]:
# 어텐션 시각화를 위한 유틸리티 함수들
class AttentionVisualizer:
    """ViT 어텐션 메커니즘 시각화 클래스"""
    
    def __init__(self, model: torch.nn.Module, device: torch.device):
        self.model = model
        self.device = device
        self.attention_maps = {}
        self.hooks = []
        self._register_hooks()
    
    def _register_hooks(self):
        """어텐션 가중치를 캡처하기 위한 훅 등록"""
        def hook_fn(name):
            def fn(module, input, output):
                # MultiheadAttention 출력에서 어텐션 가중치 추출
                if hasattr(module, 'attention_weights'):
                    self.attention_maps[name] = module.attention_weights.detach()
            return fn
        
        # 모든 어텐션 블록에 훅 등록
        for i, block in enumerate(self.model.blocks):
            hook = block.attn.register_forward_hook(hook_fn(f'layer_{i}'))
            self.hooks.append(hook)
    
    def get_attention_rollout(self, attentions: torch.Tensor, head_fusion: str = 'mean') -> torch.Tensor:
        """
        Attention Rollout 계산
        Args:
            attentions: [num_layers, num_heads, seq_len, seq_len]
            head_fusion: 멀티헤드 융합 방식 ('mean', 'max', 'min')
        Returns:
            rollout: [seq_len, seq_len]
        """
        batch_size, num_heads, seq_len, _ = attentions[0].shape
        
        # 헤드별 어텐션 융합
        if head_fusion == 'mean':
            attentions = [att.mean(dim=1) for att in attentions]  # [batch, seq_len, seq_len]
        elif head_fusion == 'max':
            attentions = [att.max(dim=1)[0] for att in attentions]
        elif head_fusion == 'min':
            attentions = [att.min(dim=1)[0] for att in attentions]
        
        # 잔차 연결을 고려한 Rollout 계산
        rollout = torch.eye(seq_len, device=self.device)
        
        for attention in attentions:
            # 잔차 연결: A' = 0.5 * A + 0.5 * I
            attention_with_residual = 0.5 * attention[0] + 0.5 * torch.eye(seq_len, device=self.device)
            rollout = torch.matmul(attention_with_residual, rollout)
        
        return rollout
    
    def visualize_attention_overlay(self, image: torch.Tensor, rollout: torch.Tensor, 
                                   patch_size: int = 16) -> np.ndarray:
        """
        어텐션 맵을 원본 이미지에 오버레이
        Args:
            image: 원본 이미지 텐서 [3, H, W]
            rollout: 어텐션 rollout [seq_len, seq_len]
            patch_size: 패치 크기
        Returns:
            overlay: 오버레이된 이미지 배열
        """
        # CLS 토큰에서 패치들로의 어텐션 추출
        cls_attention = rollout[0, 1:]  # CLS -> patches
        
        # 패치 그리드 크기 계산
        h_patches = w_patches = int(np.sqrt(len(cls_attention)))
        
        # 어텐션을 2D로 재구성
        attention_2d = cls_attention.reshape(h_patches, w_patches).cpu().numpy()
        
        # 원본 이미지 크기로 업샘플링
        img_size = image.shape[-1]
        attention_resized = cv2.resize(attention_2d, (img_size, img_size))
        
        # 정규화 (0-1 범위)
        attention_resized = (attention_resized - attention_resized.min()) / \
                           (attention_resized.max() - attention_resized.min())
        
        # 이미지를 numpy 배열로 변환
        img_np = image.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
        
        # 컬러맵 적용 (jet 컬러맵)
        attention_colored = plt.cm.jet(attention_resized)[:, :, :3]
        
        # 오버레이 (알파 블렌딩)
        alpha = 0.5
        overlay = alpha * attention_colored + (1 - alpha) * img_np
        
        return (overlay * 255).astype(np.uint8)
    
    def cleanup(self):
        """등록된 훅 제거"""
        for hook in self.hooks:
            hook.remove()
        self.attention_maps.clear()

print("✅ 어텐션 시각화 클래스 정의 완료")


In [ ]:
# 샘플 이미지 생성 및 로드
def create_sample_image() -> torch.Tensor:
    """테스트용 샘플 이미지 생성 (그라디언트 패턴)"""
    # 224x224 RGB 이미지 생성
    img = np.zeros((224, 224, 3), dtype=np.uint8)
    
    # 중앙에 원형 패턴 생성
    center = (112, 112)
    for y in range(224):
        for x in range(224):
            dist = np.sqrt((x - center[0])**2 + (y - center[1])**2)
            angle = np.arctan2(y - center[1], x - center[0])
            
            # 동심원과 방사형 패턴 조합
            r = int(127 * (1 + np.sin(dist * 0.1)) / 2)
            g = int(127 * (1 + np.sin(angle * 3)) / 2)
            b = int(127 * (1 + np.cos(dist * 0.05 + angle)) / 2)
            
            img[y, x] = [r, g, b]
    
    return img

# 샘플 이미지 생성
print("🎨 샘플 이미지 생성 중...")
sample_img = create_sample_image()

# PIL 이미지로 변환 후 전처리
pil_img = Image.fromarray(sample_img)
input_tensor = transforms(pil_img).unsqueeze(0).to(device)

print(f"✅ 입력 텐서 생성 완료: {input_tensor.shape}")

# 원본 이미지 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 원본 이미지
axes[0].imshow(sample_img)
axes[0].set_title('원본 샘플 이미지', fontsize=14, fontweight='bold')
axes[0].axis('off')

# 전처리된 이미지 (정규화 해제)
preprocessed = input_tensor[0].cpu()
# ImageNet 정규화 해제
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
unnorm = preprocessed * std + mean
unnorm = torch.clamp(unnorm, 0, 1)

axes[1].imshow(unnorm.permute(1, 2, 0))
axes[1].set_title('전처리된 이미지', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/sample_image.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 샘플 이미지 저장 완료: reports/figures/sample_image.png")


In [ ]:
# ViT 추론 실행 및 어텐션 캡처
print("🔮 ViT 추론 실행 중...")

# 어텐션 시각화 도구 초기화
visualizer = AttentionVisualizer(model, device)

# 간단한 어텐션 캡처를 위한 수정된 forward 함수
def forward_with_attention(model, x):
    """어텐션 가중치를 캡처하는 forward 함수"""
    attentions = []
    
    # 패치 임베딩
    x = model.patch_embed(x)
    
    # CLS 토큰 추가
    cls_token = model.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)
    
    # 위치 임베딩 추가
    x = model.pos_drop(x + model.pos_embed)
    
    # 각 트랜스포머 블록 실행하며 어텐션 캡처
    for i, block in enumerate(model.blocks):
        # 어텐션 계산 (수정된 방식)
        B, N, C = x.shape
        qkv = block.attn.qkv(block.norm1(x)).reshape(B, N, 3, block.attn.num_heads, C // block.attn.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        
        # 어텐션 스코어 계산
        attn = (q @ k.transpose(-2, -1)) * block.attn.scale
        attn = attn.softmax(dim=-1)
        attentions.append(attn.detach())
        
        # 나머지 블록 계산
        x = x + block.attn.proj(block.attn.proj_drop((attn @ v).transpose(1, 2).reshape(B, N, C)))
        x = x + block.mlp(block.norm2(x))
    
    # 최종 정규화 및 분류
    x = model.norm(x)
    logits = model.head(x[:, 0])  # CLS 토큰만 사용
    
    return logits, attentions

# 추론 실행
with torch.no_grad():
    logits, attentions = forward_with_attention(model, input_tensor)

# 예측 결과 분석
probs = F.softmax(logits, dim=1)
top_k = 5
top_probs, top_indices = torch.topk(probs, top_k, dim=1)

print(f"✅ 추론 완료! Top-{top_k} 예측 결과:")
print(f"🎯 예측 확률 분포 - 엔트로피: {torch.distributions.Categorical(probs).entropy().item():.3f}")

# Top-K 결과 출력
results_data = []
for i in range(top_k):
    class_idx = top_indices[0, i].item()
    prob = top_probs[0, i].item()
    class_name = imagenet_classes[class_idx]
    
    print(f"  {i+1}. {class_name:<30} ({prob:.1%})")
    results_data.append({
        'rank': i+1,
        'class_idx': class_idx,
        'class_name': class_name,
        'probability': prob,
        'confidence': prob
    })

# 결과를 DataFrame으로 저장
results_df = pd.DataFrame(results_data)
results_df.to_csv('../reports/tables/preds_topk.csv', index=False)
print(f"\n💾 예측 결과 저장: reports/tables/preds_topk.csv")

# 어텐션 통계
print(f"\n📊 어텐션 정보:")
print(f"   레이어 수: {len(attentions)}")
print(f"   어텐션 헤드 수: {attentions[0].shape[1]}")
print(f"   시퀀스 길이: {attentions[0].shape[2]}")
print(f"   패치 수: {attentions[0].shape[2] - 1}")  # CLS 토큰 제외


In [ ]:
# 어텐션 Rollout 계산 및 시각화
print("🎨 어텐션 시각화 생성 중...")

# Attention Rollout 계산
rollout = visualizer.get_attention_rollout(attentions, head_fusion='mean')

# 어텐션 오버레이 생성
overlay = visualizer.visualize_attention_overlay(
    input_tensor[0], rollout, patch_size=16
)

# 패치 그리드 시각화
def visualize_patch_grid(image: torch.Tensor, patch_size: int = 16) -> np.ndarray:
    """패치 분할을 시각화"""
    img_np = image.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    
    h, w = img_np.shape[:2]
    grid_img = img_np.copy()
    
    # 세로 선 그리기
    for x in range(0, w, patch_size):
        grid_img[:, x:x+2] = [1, 0, 0]  # 빨간색 선
    
    # 가로 선 그리기
    for y in range(0, h, patch_size):
        grid_img[y:y+2, :] = [1, 0, 0]  # 빨간색 선
    
    return (grid_img * 255).astype(np.uint8)

patch_grid = visualize_patch_grid(input_tensor[0])

# 종합 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 원본 이미지
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('원본 이미지', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# 패치 그리드
axes[0, 1].imshow(patch_grid)
axes[0, 1].set_title(f'패치 분할 (16×16)', fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

# 어텐션 맵 (CLS → patches)
cls_attention = rollout[0, 1:].reshape(14, 14).cpu().numpy()
im1 = axes[1, 0].imshow(cls_attention, cmap='viridis', interpolation='bilinear')
axes[1, 0].set_title('CLS 토큰 어텐션 맵', fontsize=14, fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im1, ax=axes[1, 0], fraction=0.046)

# 어텐션 오버레이
axes[1, 1].imshow(overlay)
axes[1, 1].set_title('어텐션 오버레이', fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/attention_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

# 패치 그리드 별도 저장
plt.figure(figsize=(8, 8))
plt.imshow(patch_grid)
plt.title('ViT 패치 분할 (16×16)', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('../reports/figures/patch_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print("💾 시각화 저장 완료:")
print("  - reports/figures/attention_overlay.png")
print("  - reports/figures/patch_grid.png")


In [ ]:
# 레이어별 어텐션 분석
print("📊 레이어별 어텐션 패턴 분석...")

# 레이어별 어텐션 다양성 계산
attention_diversity = []
attention_sparsity = []

for i, attn in enumerate(attentions):
    # 평균 어텐션 (헤드별 평균)
    mean_attn = attn.mean(dim=1)[0]  # [seq_len, seq_len]
    
    # CLS 토큰에서 패치로의 어텐션
    cls_to_patches = mean_attn[0, 1:]
    
    # 다양성 측정 (엔트로피)
    diversity = torch.distributions.Categorical(cls_to_patches).entropy().item()
    attention_diversity.append(diversity)
    
    # 희소성 측정 (상위 10% 패치가 차지하는 어텐션 비율)
    k = int(0.1 * len(cls_to_patches))
    top_k_sum = torch.topk(cls_to_patches, k)[0].sum().item()
    sparsity = top_k_sum
    attention_sparsity.append(sparsity)

# 레이어별 분석 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 어텐션 다양성 (엔트로피)
axes[0].plot(range(1, len(attention_diversity) + 1), attention_diversity, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('레이어 번호', fontsize=12)
axes[0].set_ylabel('어텐션 엔트로피', fontsize=12)
axes[0].set_title('레이어별 어텐션 다양성', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 어텐션 희소성
axes[1].plot(range(1, len(attention_sparsity) + 1), attention_sparsity, 's-', 
             color='orange', linewidth=2, markersize=8)
axes[1].set_xlabel('레이어 번호', fontsize=12)
axes[1].set_ylabel('상위 10% 패치 어텐션 합', fontsize=12)
axes[1].set_title('레이어별 어텐션 집중도', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 첫 번째와 마지막 레이어 어텐션 비교
first_layer_attn = attentions[0].mean(dim=1)[0, 0, 1:].reshape(14, 14).cpu().numpy()
last_layer_attn = attentions[-1].mean(dim=1)[0, 0, 1:].reshape(14, 14).cpu().numpy()

# 히트맵 비교
diff_attn = last_layer_attn - first_layer_attn
im = axes[2].imshow(diff_attn, cmap='RdBu_r', vmin=-diff_attn.max(), vmax=diff_attn.max())
axes[2].set_title('어텐션 변화 (마지막 - 첫 번째)', fontsize=14, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('../reports/figures/attention_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# 통계 요약
print(f"\n📈 어텐션 패턴 통계:")
print(f"   초기 레이어 엔트로피: {attention_diversity[0]:.3f}")
print(f"   최종 레이어 엔트로피: {attention_diversity[-1]:.3f}")
print(f"   엔트로피 변화: {attention_diversity[-1] - attention_diversity[0]:+.3f}")
print(f"   평균 집중도: {np.mean(attention_sparsity):.3f}")

# 정리
visualizer.cleanup()
print("\n✅ 어텐션 분석 완료!")


## 📋 실험 결과 요약

**주요 산출물**:
1. **preds_topk.csv**: Top-5 예측 결과 및 확신도
2. **attention_overlay.png**: 어텐션 집중 영역 시각화
3. **patch_grid.png**: ViT 패치 분할 시각화
4. **attention_analysis.png**: 레이어별 어텐션 패턴 분석

**핵심 인사이트**:
- ViT는 입력 이미지를 16×16 패치로 분할하여 처리
- 어텐션 메커니즘을 통해 중요한 영역에 집중
- 레이어가 깊어질수록 어텐션 패턴이 더 집중적으로 변화
- CLS 토큰이 분류에 필요한 전역 정보를 효과적으로 수집

**다음 단계**: 직사각형 입력에서의 성능 분석 (`02_rectangular_inputs.ipynb`)
